# Restaurant (Fodor’s–Zagat’s) Dataset — Deep Dive

> A classic **Entity Resolution / Record Linkage** benchmark built from two restaurant guides: **Fodor’s** and **Zagat’s**.  
> This package contains both **unsegmented** (raw text) and **segmented** (fielded) versions, plus ground truth for matches.

---

## 1) What this dataset is for

The goal is to identify which records from **two different sources** refer to the **same real-world restaurant**.

- Source A: **Fodor’s** guide records  
- Source B: **Zagat’s** guide records  
- Task: find the **true matching pairs** across sources (entity linkage / record linkage).

A key property is **extreme class imbalance** in pair space (very few true matches among all possible pairs), which makes **blocking/candidate generation** essential for practical ER experiments.

---

## 2) Files included in this package (from `restaurant.tar.gz`)

This archive contains:

### Unsegmented (raw) versions
- `restaurant/original/fodors.txt` — **533** Fodor’s records (one record per line, raw / unsegmented)
- `restaurant/original/zagats.txt` — **331** Zagat’s records (one record per line, raw / unsegmented)
- `restaurant/original/match-pairs.txt` — **112** gold matching pairs (paired raw entries separated by `########################`)

### Segmented (fielded) versions
- `restaurant/fz.arff` — **864** segmented records (**includes phone**)
- `restaurant/fz-nophone.arff` — **864** segmented records (**phone removed**)  
- `restaurant/README` — package notes (incl. warning that *phone makes the task too easy*)

Sanity check: **533 + 331 = 864** total records.

---

## 3) Schema of the segmented data (`.arff`)

### `fz.arff` columns
- `name` (string)
- `addr` (string)
- `city` (string)
- `phone` (string)
- `type` (string; cuisine/category)
- `class` (integer)

### `fz-nophone.arff` columns
- `name`
- `addr`
- `city`
- `type`
- `class`

✅ Important: **`class` is NOT a binary label (match/non-match)**.  
It is an **entity/cluster identifier**: records with the same `class` refer to the same real-world restaurant.

That means this dataset supports:
- **pairwise linkage evaluation** (derive positive pairs from same-`class` with size 2)
- **clustering evaluation** (connected components / union-find / assignment)

---

## 4) Ground truth structure (what “112 duplicates” means here)

From `fz-nophone.arff` (computed from your archive):

- Total records: **864**
- Distinct real-world restaurants (clusters): **752**
- Cluster size distribution:
  - **640** clusters of size **1** (restaurant appears in only one source)
  - **112** clusters of size **2** (restaurant appears in both sources) ✅

So the “**112 duplicates**” correspond exactly to **112 matched entities**, each with **two records (one per guide)**.

### Class ID ranges (observed in this package)
- `class = 0..111` → always size **2** (the 112 true cross-source matches)
- `class = 112..533` → size **1** (Fodor-only singletons)
- `class = 534..752` → size **1** (Zagat-only singletons)

This makes the dataset essentially a **bipartite linkage** problem with at most **1–1** matching per entity.

---

## 5) Pair-space imbalance (why blocking matters)

If you compare **every** Fodor record with **every** Zagat record:

\[
533 \times 331 = 176{,}423 \text{ candidate pairs}
\]

True matches: **112**

Positive rate:
\[
\frac{112}{176{,}423} \approx 0.0635\%
\]

That’s why any serious ER pipeline needs **candidate generation** (blocking) before scoring.

---

## 6) The “phone makes it trivial” issue

Using `fz.arff` (with phone), for the **112** matched pairs:

- After normalizing phone to digits-only (`remove_non_digits`), **108/112** pairs have **identical phone digits**.
- Only **4** matched pairs have different digit strings.

This is why `README` recommends using **`fz-nophone.arff`** for meaningful algorithm comparisons.

---

## 7) Similarity behavior on the true matched pairs (no-phone version)

On the **112** gold matched pairs, a simple sequence-based string similarity typically shows:

- `name` similarity: high on average (many pairs near 1.0, but with hard outliers)
- `addr` similarity: high but with format noise (abbreviations, punctuation, missing pieces)
- `city` similarity: good but needs normalization (`LA` vs `Los Angeles`, `W. Hollywood` vs `Hollywood`)
- `type` similarity: usually the weakest / noisiest (categories differ across guides)

Takeaway:
- **Name + address + city** carry most of the matching signal.
- Treat `type` as a **soft feature** (low weight or robust encoding).

---

## 8) Segmented vs. unsegmented: what changes?

### Segmented (`.arff`)
You can immediately do ER with string similarity, token similarity, ML models, etc.

### Unsegmented (`original/*.txt`)
Records are raw text lines where name/address/city/phone/type can be interleaved and irregular.
This adds an extra task:
- **record parsing / field extraction**
before you can run the ER/linkage pipeline.

That makes the unsegmented version useful for studying **end-to-end** ER:
`parse → normalize → block → score → decide`

---

## 9) Recommended experimental setups

### Setup A — clean ER benchmark (recommended)
Use **`fz-nophone.arff`**.

Pipeline:
1. **Normalization**
   - lowercasing, whitespace collapse
   - standardize abbreviations: `st` ↔ `street`, `ave` ↔ `avenue`, etc.
2. **Blocking (candidate generation)**
   - examples: block by normalized `city`, or `(city + first token of name)`, or phonetic keys
3. **Pair scoring**
   - Jaro-Winkler / Levenshtein / token Jaccard / TF-IDF cosine on `name`, `addr`
   - lighter weight features on `city`, `type`
4. **Decision**
   - thresholding OR (better) **maximum bipartite matching** per block
5. **Evaluation**
   - Pairwise Precision / Recall / F1 on the 112 gold matches

### Setup B — “cheating” upper bound
Use **`fz.arff`** with phone digits normalization.  
This should yield near-perfect linkage and is mainly useful as a sanity check.

---

## 10) Plots generated (from your archive)

The following plots were produced from your `restaurant.tar.gz`:

1. **Cluster-size distribution**: only sizes **1** and **2** (640 vs 112).
2. **Matched-pair similarity histograms** (112 pairs):
   - `name` similarity distribution
   - `addr` similarity distribution
   - `type` similarity distribution
3. **Phone-equality bar chart** (112 pairs):
   - 108 True vs 4 False after digits-only normalization

---

## 11) Quick “gotchas”

- **`class` is an entity ID**, not binary labels.
- **Phone dominates** if included.
- `type` may hurt if weighted too heavily.
- City names vary; normalize aggressively.
- For fair comparison across methods, report both:
  - *candidate pairs evaluated* (efficiency)
  - *final PR/F1* (effectiveness)

---

## 12) Suggested baselines (good for papers)

- **Rule-based**: name+addr token overlap + city block
- **Probabilistic**: Fellegi–Sunter style scoring (log-likelihood ratios)
- **ML**: logistic regression / gradient boosting over string-sim features
- **Assignment-based**: maximum bipartite matching within blocks (prevents many-to-one errors)

---

In [1]:
!tree data

data
├── fz.arff
├── fz-nophone.arff
├── original
│   ├── fodors.txt
│   ├── match-pairs.txt
│   └── zagats.txt
└── README

2 directories, 6 files
